# Clase 097 — Agglomerative, BIRCH, Mean Shift, Affinity Propagation, Spectral

El "zoológico" de clustering más allá de K-Means y DBSCAN. Comparamos jerárquico (dendrograma), escalable (BIRCH), denso sin `k` (Mean Shift), exemplars (Affinity Propagation) y grafo espectral.

Requiere: `numpy`, `scipy`, `scikit-learn`, `matplotlib`.

## 🧠 Intuición previa

Clave para leer todo lo que sigue: **no existe un clustering "correcto"**. Cada algoritmo asume una
**forma distinta de grupo** — K-Means y Mean Shift esperan blobs redondos, DBSCAN/HDBSCAN buscan
regiones densas de cualquier forma, el aglomerativo arma una jerarquía, Spectral corta un grafo de
vecinos. Por eso la tabla comparativa del final no es decorativa: te dice **cuál usar según cómo son
tus datos** (esféricos, por densidad, anidados, jerárquicos), no cuál es "el mejor".

## 1. Dendrograma jerárquico (`ward`)

`scipy.cluster.hierarchy.linkage` construye la jerarquía bottom-up; cortarla a una altura da los clusters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.datasets import make_blobs

np.random.seed(42)
Xs, _ = make_blobs(n_samples=50, centers=4, cluster_std=0.6, random_state=42)

Z = linkage(Xs, method="ward")
labels_cut = fcluster(Z, t=4, criterion="maxclust")
print("clusters tras cortar a 4:", np.unique(labels_cut))
assert len(np.unique(labels_cut)) == 4

plt.figure(figsize=(9, 4))
dendrogram(Z, truncate_mode="lastp", p=20)
plt.title("Dendrograma (ward linkage)")
plt.xlabel("muestras"); plt.ylabel("distancia de fusion")
plt.tight_layout(); plt.show()

## 2. Agglomerative y comparación de `linkage`

Sobre `make_moons`, `single` linkage recupera las lunas (sigue la densidad); `ward` las parte.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score

Xm, ym = make_moons(n_samples=300, noise=0.05, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for a, link in zip(axes, ("ward", "complete", "average", "single")):
    lab = AgglomerativeClustering(n_clusters=2, linkage=link).fit_predict(Xm)
    a.scatter(Xm[:, 0], Xm[:, 1], c=lab, cmap="coolwarm", s=10)
    a.set_title(f"{link}\nARI={adjusted_rand_score(ym, lab):.2f}")
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("Linkage comparison sobre make_moons")
plt.tight_layout(); plt.show()

## 3. BIRCH: escalable

Recorre los datos una vez construyendo un CF-tree. `threshold` controla el radio de los sub-clusters.

In [ ]:
from sklearn.cluster import Birch, KMeans
import time

Xb, yb = make_blobs(n_samples=20000, centers=5, cluster_std=0.7, random_state=42)

t0 = time.perf_counter()
birch = Birch(n_clusters=5, threshold=0.5).fit(Xb)
t_birch = time.perf_counter() - t0

t0 = time.perf_counter()
km = KMeans(n_clusters=5, n_init=10, random_state=42).fit(Xb)
t_km = time.perf_counter() - t0

print(f"BIRCH : {t_birch*1000:6.1f} ms | ARI {adjusted_rand_score(yb, birch.labels_):.3f}")
print(f"KMeans: {t_km*1000:6.1f} ms | ARI {adjusted_rand_score(yb, km.labels_):.3f}")

for thr in (0.1, 0.5, 1.0):
    b = Birch(n_clusters=None, threshold=thr).fit(Xb)
    print(f"threshold={thr}: {len(np.unique(b.labels_))} sub-clusters en el CF-tree")

## 4. Mean Shift: descubre `k` solo

Cada punto trepa al modo de densidad. `estimate_bandwidth` da un ancho de kernel de arranque.

In [ ]:
from sklearn.cluster import MeanShift, estimate_bandwidth

Xms, _ = make_blobs(n_samples=600, centers=3, cluster_std=0.6, random_state=42)

bw = estimate_bandwidth(Xms, quantile=0.2, n_samples=300, random_state=42)
ms = MeanShift(bandwidth=bw, bin_seeding=True).fit(Xms)
k_encontrado = len(np.unique(ms.labels_))
print(f"bandwidth (quantile=0.2) = {bw:.3f} -> {k_encontrado} clusters")
assert k_encontrado == 3, "Mean Shift deberia recuperar 3 clusters"

bw2 = estimate_bandwidth(Xms, quantile=0.5, n_samples=300, random_state=42)
k2 = len(np.unique(MeanShift(bandwidth=bw2, bin_seeding=True).fit(Xms).labels_))
print(f"bandwidth (quantile=0.5) = {bw2:.3f} -> {k2} clusters (colapsa)")

plt.figure(figsize=(7, 5))
plt.scatter(Xms[:, 0], Xms[:, 1], c=ms.labels_, cmap="tab10", s=10)
plt.scatter(ms.cluster_centers_[:, 0], ms.cluster_centers_[:, 1],
            c="black", marker="X", s=150)
plt.title(f"Mean Shift: {k_encontrado} modos de densidad")
plt.tight_layout(); plt.show()

## 5. Spectral vs K-Means en `make_circles`

Spectral usa autovectores del grafo de similitud: separa círculos concéntricos donde K-Means falla.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.cluster import SpectralClustering

Xc, yc = make_circles(n_samples=500, factor=0.5, noise=0.05, random_state=42)

lab_km = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Xc)
lab_sp = SpectralClustering(n_clusters=2, affinity="nearest_neighbors",
                            n_neighbors=10, random_state=42).fit_predict(Xc)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(Xc[:, 0], Xc[:, 1], c=lab_km, cmap="coolwarm", s=12)
ax[0].set_title(f"K-Means (ARI={adjusted_rand_score(yc, lab_km):.2f})")
ax[1].scatter(Xc[:, 0], Xc[:, 1], c=lab_sp, cmap="coolwarm", s=12)
ax[1].set_title(f"Spectral (ARI={adjusted_rand_score(yc, lab_sp):.2f})")
plt.tight_layout(); plt.show()
assert adjusted_rand_score(yc, lab_sp) > adjusted_rand_score(yc, lab_km)

## 6. Comparativa con Affinity Propagation sobre Iris

Corremos los 5 métodos sobre `load_iris` y medimos Adjusted Rand Index contra las etiquetas reales.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.cluster import AffinityPropagation
from sklearn.preprocessing import StandardScaler

Xi, yi = load_iris(return_X_y=True)
Xi_s = StandardScaler().fit_transform(Xi)

modelos = {
    "Agglomerative": AgglomerativeClustering(n_clusters=3, linkage="ward"),
    "BIRCH": Birch(n_clusters=3, threshold=0.5),
    "MeanShift": MeanShift(bandwidth=estimate_bandwidth(Xi_s, quantile=0.3, random_state=42)),
    "AffinityProp": AffinityPropagation(damping=0.9, random_state=42),
    "Spectral": SpectralClustering(n_clusters=3, affinity="nearest_neighbors",
                                   n_neighbors=10, random_state=42),
}
print(f"{'metodo':>15} {'ARI':>7}")
for name, m in modelos.items():
    lab = m.fit_predict(Xi_s)
    print(f"{name:>15} {adjusted_rand_score(yi, lab):>7.3f}")
print("\nAgglomerative-ward y Spectral suelen superar 0.5-0.7 en Iris.")

## Ejercicios

1. Generá 50 puntos en 4 blobs, calculá `linkage(X, 'ward')`, graficá el dendrograma y compará con `AgglomerativeClustering(n_clusters=4)`.
2. Sobre `make_moons`, ajustá los 4 linkages y explicá cuál recupera las lunas y por qué.
3. Sobre 3 blobs, corré `MeanShift` con `estimate_bandwidth` (quantile 0.2 y 0.5) y observá cómo colapsa.
4. Sobre `make_circles`, compará `KMeans` vs `SpectralClustering(affinity='nearest_neighbors')`.

## Conclusiones

- Agglomerative da una jerarquía completa (dendrograma); el `linkage` decide la forma de los clusters (`single` sufre chaining).
- BIRCH es one-pass y escala a millones de filas; Mean Shift descubre `k` solo vía la densidad.
- Affinity Propagation elige exemplars reales pero es O(n²): solo para datasets chicos.
- Spectral clustering recupera clusters no convexos usando el grafo Laplaciano; usá `affinity='nearest_neighbors'` para evitar tunear `gamma`.

## ✅ Soluciones de los ejercicios

Cinco ejercicios del *clustering zoo*: dendrograma jerárquico, comparación de `linkage`, BIRCH escalable, Mean Shift (descubre `k` solo) y Spectral vs K-Means en círculos. Reducimos el tamaño del test de BIRCH para correr en < 60s. `n_jobs=1`.

**Ejercicio 1 — Dendrograma (`ward`).** `linkage` + corte a 4 clusters; comparamos con `AgglomerativeClustering(n_clusters=4)`.

In [ ]:
import numpy as np, time, matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import (AgglomerativeClustering, Birch, KMeans, MeanShift,
                             estimate_bandwidth, SpectralClustering)
from sklearn.metrics import adjusted_rand_score

X, ytrue = make_blobs(n_samples=50, centers=4, random_state=42)
Z = linkage(X, method='ward')
plt.figure(figsize=(8, 3.5)); dendrogram(Z, no_labels=True)
plt.title('Dendrograma (ward)'); plt.tight_layout(); plt.show()
lab_cut = fcluster(Z, t=4, criterion='maxclust')
lab_agg = AgglomerativeClustering(n_clusters=4, linkage='ward').fit_predict(X)
print('ARI corte-dendrograma vs Agglomerative:',
      round(adjusted_rand_score(lab_cut, lab_agg), 3))
assert adjusted_rand_score(lab_cut, lab_agg) > 0.9, 'deben coincidir: es el mismo algoritmo'

**Ejercicio 2 — Comparación de `linkage`.** Sobre moons, solo `single` sigue la forma no convexa (encadena por vecindad); `ward`/`complete`/`average` la parten.

In [ ]:
Xm, _ = make_moons(n_samples=300, noise=0.05, random_state=42)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, link in zip(axes, ['ward', 'complete', 'average', 'single']):
    lab = AgglomerativeClustering(n_clusters=2, linkage=link).fit_predict(Xm)
    ax.scatter(Xm[:, 0], Xm[:, 1], c=lab, cmap='coolwarm', s=10)
    ax.set_title(link)
plt.tight_layout(); plt.show()
print('single linkage recupera las dos lunas; los demas cortan por compacidad.')

**Ejercicio 3 — BIRCH escalable.** Con 30k puntos (reducido para el runtime) comparamos tiempo vs K-Means y variamos `threshold` para ver los sub-clusters del CF-tree.

In [ ]:
Xbig, _ = make_blobs(n_samples=30000, centers=5, random_state=42)
def timed(model):
    t0 = time.perf_counter(); model.fit(Xbig); return time.perf_counter() - t0
t_birch = timed(Birch(n_clusters=5, threshold=0.5))
t_km = timed(KMeans(n_clusters=5, n_init=3, random_state=42))
print(f'BIRCH : {t_birch:.2f}s | KMeans: {t_km:.2f}s')
for th in [0.1, 0.5, 1.0]:
    b = Birch(n_clusters=5, threshold=th).fit(Xbig)
    print(f'  threshold={th:<4} -> {len(b.subcluster_centers_)} sub-clusters en el CF-tree')
print('threshold chico = CF-tree fino (mas sub-clusters); grande = mas grueso.')

**Ejercicio 4 — Mean Shift sin saber `k`.** `estimate_bandwidth` fija el radio; `quantile` chico recupera 3 clusters, grande colapsa.

In [ ]:
Xb3, _ = make_blobs(n_samples=500, centers=3, cluster_std=0.7, random_state=42)
for q in [0.2, 0.5]:
    bw = estimate_bandwidth(Xb3, quantile=q)
    ms = MeanShift(bandwidth=bw).fit(Xb3)
    print(f'quantile={q} -> bandwidth {bw:.2f} -> {len(set(ms.labels_))} clusters')
print('Mean Shift no pide k: el bandwidth (via quantile) decide cuantos modos encuentra.')

**Ejercicio 5 — Spectral vs K-Means en `make_circles`.** Dos círculos concéntricos: Spectral (grafo de vecinos) los separa; K-Means no.

In [ ]:
Xc, _ = make_circles(n_samples=500, factor=0.5, noise=0.05, random_state=42)
lab_km = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Xc)
lab_sp = SpectralClustering(n_clusters=2, affinity='nearest_neighbors',
                            random_state=42).fit_predict(Xc)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(Xc[:, 0], Xc[:, 1], c=lab_km, cmap='coolwarm', s=10); ax[0].set_title('K-Means')
ax[1].scatter(Xc[:, 0], Xc[:, 1], c=lab_sp, cmap='coolwarm', s=10); ax[1].set_title('Spectral')
plt.tight_layout(); plt.show()
print('Spectral trabaja sobre el grafo de similitud: separa formas no convexas anidadas.')